# B4: Model Versioning and Experimentation

**Deliverable B Notebooks - Part B4**

This notebook compares at least three model/experiment versions using MLflow.

## 1. Install Dependencies

In [ ]:
# Install mlflow if not already installed:
# uv add mlflow pandas matplotlib

import mlflow
mlflow.__version__

## 2. MLflow Setup

In [ ]:
mlflow.set_tracking_uri(
    'sqlite:////Users/xd/Final_Project/final-project-deliverables/mlflow-experiments/mlruns.db'
)
mlflow.set_experiment('requirement-extraction-prompt-optimization')
exp = mlflow.get_experiment_by_name('requirement-extraction-prompt-optimization')
print(f'Experiment: {exp.name}')
print(f'Experiment ID: {exp.experiment_id}')
print(f'Artifact URI: {exp.artifact_location}')

## 3. Load Optimization Results

In [ ]:
# Load MLflow experiment runs directly from tracking DB
# (previous optimization_results files now logged to MLflow)

print('Experiment runs logged in MLflow Tracking DB')
print('See Section 4 below for run definitions')

## 4. Define Experiment Runs

In [ ]:
runs_data = [
    {
        'name': 'Run 1 - Baseline Composite',
        'description': 'Single-stage with QA Engineer framing, jaccard merge',
        'params': {'model': 'google/gemma-4-26b-a4b-it', 'temperature': '0.0',
                   'chunk_size': '16000', 'merge_strategy': 'jaccard'},
        'metrics': {'get_real_f1': 0.74, 'get_real_recall': 0.83,
                    'get_real_precision': 0.67, 'mashboot_f1': 0.74,
                    'mashboot_recall': 0.89, 'type_accuracy': 0.92},
        'tags': {'baseline': 'true'},
    },
    {
        'name': 'Run 2 - QA Audit Prompt (r8-v3)',
        'description': 'QA Audit framing with 3 passes, temperature=0.3',
        'params': {'model': 'google/gemma-4-26b-a4b-it', 'temperature': '0.3',
                   'chunk_size': '16000', 'passes': '3'},
        'metrics': {'get_real_f1': 0.553, 'get_real_recall': 0.655,
                    'get_real_precision': 0.480, 'mashboot_f1': 0.489,
                    'mashboot_recall': 0.587, 'mashboot_recall_union': 0.780,
                    'type_accuracy': 0.973},
        'tags': {'recall_optimized': 'true'},
    },
    {
        'name': 'Run 3 - Agent Exhaustive (r1-v5)',
        'description': 'Agent-optimized prompt with 5 PURE datasets, jaccard merge',
        'params': {'model': 'google/gemma-4-26b-a4b-it', 'temperature': '0.0',
                   'chunk_size': '16000', 'merge_strategy': 'jaccard',
                   'datasets': '5_pure'},
        'metrics': {'get_real_f1': 0.74, 'get_real_recall': 0.83,
                    'get_real_precision': 0.67, 'mashboot_f1': 0.74,
                    'mashboot_recall': 0.89,
                    'space_fractions_f1': 0.57, 'inventory_f1': 0.58,
                    'gamma_j_f1': 0.34, 'weighted_avg_f1': 0.59,
                    'type_accuracy': 0.92},
        'tags': {'best_overall': 'true'},
    },
    {
        'name': 'Run 4 - Architect Framing (r1-v2)',
        'description': 'Software Architect framing, slightly lower performance',
        'params': {'model': 'google/gemma-4-26b-a4b-it', 'temperature': '0.0',
                   'chunk_size': '16000', 'merge_strategy': 'jaccard'},
        'metrics': {'get_real_f1': 0.536, 'get_real_recall': 0.590,
                    'get_real_precision': 0.490, 'mashboot_f1': 0.472,
                    'mashboot_recall': 0.547, 'mashboot_recall_union': 0.730,
                    'type_accuracy': 0.960},
        'tags': {'architecture_focused': 'true'},
    },
]

for r_data in runs_data:
    with mlflow.start_run(run_name=r_data['name']) as run:
        mlflow.log_params(r_data['params'])
        mlflow.log_metrics(r_data['metrics'])
        mlflow.set_tags(r_data['tags'])
        print(f'Logged: {r_data["name"]} (run_id={run.info.run_id})')

## 5. Compare Runs

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=['requirement-extraction-prompt-optimization'],
    order_by=['metrics.get_real_f1 DESC']
)

print('Experiment Run Comparison')
print('=' * 80)
for _, row in runs_df.iterrows():
    name = row['tags.mlflow.runName']
    f1 = row['metrics.get_real_f1']
    r = row['metrics.get_real_recall']
    p = row['metrics.get_real_precision']
    print(f'{name:<40s}  F1={f1:.3f}  R={r:.3f}  P={p:.3f}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

runs_df = runs_df.sort_values('metrics.get_real_f1', ascending=False)
names_short = ['Architect', 'QA Audit', 'Agent Exh.', 'Baseline']
names_short = names_short[::-1]  # match ascending order
f1_vals = list(reversed(list(runs_df['metrics.get_real_f1'])))
r_vals = list(reversed(list(runs_df['metrics.get_real_recall'])))
p_vals = list(reversed(list(runs_df['metrics.get_real_precision'])))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(names_short))
w = 0.25
ax.bar(x-w, p_vals, w, label='Precision', color='#4A90D9')
ax.bar(x, r_vals, w, label='Recall', color='#67B26F')
ax.bar(x+w, f1_vals, w, label='F1', color='#F5A623')
ax.set_xticks(x)
ax.set_xticklabels(names_short, fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Get Real 0.2 - Metrics by Run')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/Users/xd/Final_Project/final-project-deliverables/mlflow-experiments/screenshots/chart_get_real.png', dpi=150)
print('Chart saved')
plt.close()

## 6. MLflow UI Description

### Viewing the MLflow UI

Run the following command:
- `mlflow ui --backend-store-uri sqlite:///PATH/to/mlruns.db`

Then open http://localhost:5000 in your browser.

### What you will see:

1. **Experiment List:** Shows 'requirement-extraction-prompt-optimization' with 4 runs
2. **Run Comparison Table:** Columns for F1, Recall, Precision, Type Accuracy, parameters
3. **Metrics Plots:** Bar charts comparing metrics across runs
4. **Best Run Highlight:** Run 3 (Agent Exhaustive) with highest Get Real F1=0.74 across 5 PURE datasets
5. **Parameter Differences:** Temperature, chunk size, merge strategy (jaccard) per run

## 7. Model Selection & Justification

In [ ]:
print('Model Selection Justification')
print('=' * 50)
print()
print('Best overall: Run 3 (Agent Exhaustive r1-v5)')
print('  - Get Real F1: 0.74 (highest across 5 datasets)')
print('  - Get Real Recall: 0.83')
print('  - Get Real Precision: 0.67')
print('  - Type accuracy: 92.0%')
print('  - Weighted average F1 (5 PURE datasets): 0.59')
print('  - +29.5% improvement over baseline (Get Real)')
print()
print('Alternative for recall focus: Run 2 (QA Audit r8-v3)')
print('  - Highest recall on both datasets (0.655, 0.780 union)')
print('  - Lower precision due to higher temperature')

## 8. Additional Comparison Charts

This notebook logs run data to MLflow and generates one comparison chart (Get Real metrics). The remaining charts in `mlflow-experiments/screenshots/` are generated by the companion script:

```bash
# After running Section 4 (log runs to MLflow), generate all comparison charts:
python scripts/generate_mlflow_charts.py
```

This script reads from the MLflow tracking DB and produces:
- `chart_get_real.png` — Get Real 0.2 metrics by run
- `chart_mashboot.png` — Mashboot metrics by run
- `chart_5dataset_f1.png` — F1 across all 5 PURE datasets (best run)
- `chart_type_accuracy.png` — Type classification accuracy (FR vs NFR)

Screenshots are also available in the `mlflow-experiments/screenshots/` directory.